<a href="https://colab.research.google.com/github/kuds/rl-car-racing/blob/main/%5BCar%20Racing%5D%20Dreamer%20World%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dreamer World Model

This notebook trains a **Dreamer-style world model** on Gymnasium's
`CarRacing-v3`, then learns a continuous-action policy entirely inside that
learned model. The implementation is a from-scratch PyTorch port that follows
the architecture introduced in DreamerV2/V3 but is intentionally trimmed so
it fits on a single L4 GPU in a few hours.

The agent has four learnable parts:

1. **Convolutional encoder/decoder** — turns 64x64 RGB frames into a 1024-d
   embedding and back.
2. **Recurrent State-Space Model (RSSM)** — a GRU-based latent dynamics model
   with a stochastic `z_t` (Gaussian) and a deterministic `h_t`. It exposes
   `observe()` for posterior inference from real frames and `imagine_step()`
   for prior rollouts in latent space.
3. **Reward / continue / decoder heads** — predict the symlog'd reward,
   episode-termination probability, and reconstructed frame from `(h, z)`.
4. **Actor / critic** — a TanhNormal policy over the 3-d continuous action and
   a value head trained on lambda-returns from imagined rollouts. The critic
   has an EMA target for bootstrap stability.

At each environment step we (a) push the transition into a sequence replay
buffer, (b) take one gradient step on the world model over a batch of
sequences, and (c) take one gradient step on the actor/critic by imagining
`H=15` steps forward from each posterior state.

**Discrete latents** (the categorical `z` from DreamerV2/V3) are *not*
implemented here for simplicity; this notebook uses Gaussian latents only.
That is the most impactful follow-up to try if you want to push the score
further.


In [ ]:
!pip install swig

In [ ]:
!pip install gymnasium[box2d] torch tqdm imageio[ffmpeg]

In [ ]:
import os
import math
import time
import platform
import random
from dataclasses import dataclass, field, asdict
from collections import deque
from importlib.metadata import version

import numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, Independent, TransformedDistribution
from torch.distributions.transforms import TanhTransform

import gymnasium
import imageio
import matplotlib
import matplotlib.pyplot
from tqdm.auto import tqdm

print(f"Python Version: {platform.python_version()}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Imageio Version: {version('imageio')}")
print(f"Swig Version: {version('swig')}")


## Config

All knobs live here. The defaults target ~6 hours of training on a single L4
GPU. Drop `total_env_steps` to do a quick smoke test, or scale the model up
(`hidden_size`, `embed_size`, `seq_len`) if you have more compute.


In [ ]:
@dataclass
class DreamerConfig:
    # Environment
    env_id: str = "CarRacing-v3"
    image_size: int = 64
    action_repeat: int = 4
    seed: int = 0

    # Replay
    buffer_capacity_episodes: int = 1000
    warmup_episodes: int = 5

    # World model architecture
    embed_size: int = 1024
    hidden_size: int = 200       # deterministic GRU state h
    stoch_size: int = 32         # Gaussian latent z
    mlp_hidden: int = 400
    min_std: float = 0.1

    # Sequence training
    batch_size: int = 16
    seq_len: int = 50
    horizon: int = 15            # imagination horizon H

    # Optimization
    lr_world: float = 3e-4
    lr_actor: float = 8e-5
    lr_critic: float = 8e-5
    grad_clip_world: float = 1000.0
    grad_clip_ac: float = 100.0
    eps: float = 1e-5

    # Losses
    kl_free_bits: float = 1.0
    kl_scale: float = 1.0
    reward_scale: float = 1.0
    continue_scale: float = 1.0
    entropy_scale: float = 3e-4

    # Returns / target
    gamma: float = 0.99
    lambda_: float = 0.95
    target_tau: float = 0.02     # polyak factor for EMA target critic

    # Training schedule
    total_env_steps: int = 400_000   # raw env steps (action_repeat counted)
    train_every_steps: int = 1       # 1:1 train ratio (per *policy* step)
    log_every_steps: int = 5_000
    eval_every_steps: int = 50_000
    eval_episodes: int = 5

    # I/O
    log_dir: str = "./logs/dreamer_car_racing"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_bf16: bool = True


config = DreamerConfig()
os.makedirs(config.log_dir, exist_ok=True)

# Performance toggles for L4
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

random.seed(config.seed)
numpy.random.seed(config.seed)
torch.manual_seed(config.seed)

device = torch.device(config.device)
print(f"Device: {device}")
print(f"Action repeat: {config.action_repeat} -> "
      f"{config.total_env_steps // config.action_repeat} policy decisions")


## Environment wrapper

Frames are downsampled to 64x64, channel-first, and normalized to `[0, 1]`.
Each policy decision is repeated `action_repeat` times, summing rewards and
forwarding any early termination.


In [ ]:
class CarRacingWrapper:
    """Light wrapper around `CarRacing-v3` for the world model.

    Returns observations as float32 tensors of shape (3, H, W) in [0, 1] and
    repeats each action `action_repeat` times.
    """

    def __init__(self, env_id, image_size=64, action_repeat=4, seed=None):
        self.env = gymnasium.make(env_id)
        self.image_size = image_size
        self.action_repeat = action_repeat
        self.action_space = self.env.action_space
        self.observation_space = self.env.observation_space
        self._seed = seed

    def _process(self, obs):
        # obs is (96, 96, 3) uint8
        img = torch.from_numpy(obs).float() / 255.0
        img = img.permute(2, 0, 1).unsqueeze(0)
        img = F.interpolate(img, size=(self.image_size, self.image_size),
                            mode="bilinear", align_corners=False)
        return img.squeeze(0).contiguous()

    def reset(self):
        if self._seed is not None:
            obs, info = self.env.reset(seed=self._seed)
            self._seed = None
        else:
            obs, info = self.env.reset()
        return self._process(obs), info

    def step(self, action):
        total_reward = 0.0
        terminated = False
        truncated = False
        info = {}
        obs = None
        for _ in range(self.action_repeat):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += float(reward)
            if terminated or truncated:
                break
        return self._process(obs), total_reward, terminated, truncated, info

    def close(self):
        self.env.close()


## Sequence replay buffer

We store complete episodes and sample `(batch_size, seq_len)` windows from
random episodes. Sequences shorter than `seq_len` are skipped.


In [ ]:
class SequenceReplayBuffer:
    """Episode-keyed replay buffer that samples fixed-length sub-sequences."""

    def __init__(self, capacity_episodes, image_size, action_dim):
        self.capacity = capacity_episodes
        self.image_size = image_size
        self.action_dim = action_dim
        self.episodes = deque(maxlen=capacity_episodes)

    def __len__(self):
        return len(self.episodes)

    def total_steps(self):
        return sum(ep["reward"].shape[0] for ep in self.episodes)

    def add_episode(self, obs, actions, rewards, continues):
        # obs:        (T+1, 3, H, W) float32 in [0, 1]
        # actions:    (T, A) float32
        # rewards:    (T,)   float32
        # continues:  (T,)   float32 (1.0 while alive, 0.0 on terminal)
        self.episodes.append({
            "obs": obs.cpu(),
            "action": actions.cpu(),
            "reward": rewards.cpu(),
            "continue": continues.cpu(),
        })

    def sample(self, batch_size, seq_len, device):
        eligible = [ep for ep in self.episodes
                    if ep["reward"].shape[0] >= seq_len]
        if not eligible:
            return None
        chosen = [random.choice(eligible) for _ in range(batch_size)]
        obs_b, act_b, rew_b, cont_b = [], [], [], []
        for ep in chosen:
            T = ep["reward"].shape[0]
            start = random.randint(0, T - seq_len)
            end = start + seq_len
            obs_b.append(ep["obs"][start:end + 1])    # T+1 frames
            act_b.append(ep["action"][start:end])
            rew_b.append(ep["reward"][start:end])
            cont_b.append(ep["continue"][start:end])
        batch = {
            "obs": torch.stack(obs_b).to(device),         # (B, T+1, 3, H, W)
            "action": torch.stack(act_b).to(device),       # (B, T, A)
            "reward": torch.stack(rew_b).to(device),       # (B, T)
            "continue": torch.stack(cont_b).to(device),    # (B, T)
        }
        return batch


## Symlog reward transform

DreamerV3 squashes rewards (and value targets) through `symlog` before the
network has to predict them. This compresses the dynamic range without losing
sign, which makes training stable across environments with very different
reward scales.


In [ ]:
def symlog(x):
    return torch.sign(x) * torch.log1p(torch.abs(x))


def symexp(x):
    return torch.sign(x) * (torch.expm1(torch.abs(x)))


## Convolutional encoder

Four stride-2 conv layers map a `(3, 64, 64)` image to a 1024-d embedding.


In [ ]:
class ConvEncoder(nn.Module):
    """4-layer stride-2 conv -> embed_size embedding for 64x64 RGB."""

    def __init__(self, embed_size=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2),       # 64 -> 31
            nn.SiLU(),
            nn.Conv2d(32, 64, 4, stride=2),      # 31 -> 14
            nn.SiLU(),
            nn.Conv2d(64, 128, 4, stride=2),     # 14 -> 6
            nn.SiLU(),
            nn.Conv2d(128, 256, 4, stride=2),    # 6  -> 2
            nn.SiLU(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 64, 64)
            flat = self.net(dummy).flatten(1).shape[1]
        self.proj = nn.Linear(flat, embed_size)
        self.embed_size = embed_size

    def forward(self, x):
        # x: (..., 3, 64, 64)
        leading = x.shape[:-3]
        x = x.reshape(-1, *x.shape[-3:])
        h = self.net(x).flatten(1)
        emb = self.proj(h)
        return emb.reshape(*leading, self.embed_size)


## Convolutional decoder

Mirror of the encoder. Takes the concatenated `(h, z)` state and
reconstructs a 64x64 RGB image. Outputs are unbounded; we use plain MSE on
raw pixel values.


In [ ]:
class ConvDecoder(nn.Module):
    """Maps (h, z) -> reconstructed 64x64 RGB image."""

    def __init__(self, in_size, hidden=256):
        super().__init__()
        self.fc = nn.Linear(in_size, hidden * 2 * 2)
        self.hidden = hidden
        self.net = nn.Sequential(
            nn.ConvTranspose2d(hidden, 128, 5, stride=2),    # 2 -> 7
            nn.SiLU(),
            nn.ConvTranspose2d(128, 64, 5, stride=2),        # 7 -> 17
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, 6, stride=2),         # 17 -> 38
            nn.SiLU(),
            nn.ConvTranspose2d(32, 3, 6, stride=2),          # 38 -> 80
        )

    def forward(self, feat):
        leading = feat.shape[:-1]
        x = self.fc(feat.reshape(-1, feat.shape[-1]))
        x = x.reshape(-1, self.hidden, 2, 2)
        x = self.net(x)
        # Center-crop to 64x64
        crop = (x.shape[-1] - 64) // 2
        x = x[..., crop:crop + 64, crop:crop + 64]
        return x.reshape(*leading, 3, 64, 64)


## Recurrent State-Space Model

The latent state is `(h_t, z_t)` where `h_t` is a deterministic GRU hidden
state and `z_t` is a stochastic Gaussian sample. Two heads operate on `h_t`:

- **prior**: predicts `p(z_t | h_t)` (used for imagination).
- **posterior**: predicts `q(z_t | h_t, e_t)` from the encoder embedding (used
  during world-model training to ground the latent in the real frame).

`observe()` runs through a real sequence step-by-step, alternating GRU update
and posterior inference. `imagine_step()` runs a single prior step given an
action, used for the actor/critic rollouts.


In [ ]:
class RSSM(nn.Module):
    """GRU-based latent dynamics with Gaussian stochastic state."""

    def __init__(self, action_dim, embed_size, hidden_size=200,
                 stoch_size=32, mlp_hidden=400, min_std=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.stoch_size = stoch_size
        self.min_std = min_std

        self.fc_input = nn.Sequential(
            nn.Linear(stoch_size + action_dim, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.SiLU(),
        )
        self.gru = nn.GRUCell(mlp_hidden, hidden_size)

        self.prior_net = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.SiLU(),
            nn.Linear(mlp_hidden, 2 * stoch_size),
        )
        self.post_net = nn.Sequential(
            nn.Linear(hidden_size + embed_size, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.SiLU(),
            nn.Linear(mlp_hidden, 2 * stoch_size),
        )

    def initial_state(self, batch_size, device):
        return {
            "h": torch.zeros(batch_size, self.hidden_size, device=device),
            "z": torch.zeros(batch_size, self.stoch_size, device=device),
        }

    def _stats_to_dist(self, stats):
        mean, std = stats.chunk(2, dim=-1)
        std = F.softplus(std) + self.min_std
        return Independent(Normal(mean, std), 1)

    def img_step(self, prev_state, prev_action):
        # Deterministic update of h, then sample prior z.
        x = torch.cat([prev_state["z"], prev_action], dim=-1)
        x = self.fc_input(x)
        h = self.gru(x, prev_state["h"])
        prior_stats = self.prior_net(h)
        prior = self._stats_to_dist(prior_stats)
        z = prior.rsample()
        return {"h": h, "z": z, "prior_stats": prior_stats}

    def obs_step(self, prev_state, prev_action, embed):
        prior = self.img_step(prev_state, prev_action)
        post_input = torch.cat([prior["h"], embed], dim=-1)
        post_stats = self.post_net(post_input)
        post = self._stats_to_dist(post_stats)
        z = post.rsample()
        return {
            "h": prior["h"],
            "z": z,
            "prior_stats": prior["prior_stats"],
            "post_stats": post_stats,
        }

    def observe(self, embeds, actions, init_state):
        """Run the posterior over a real sequence.

        Args:
            embeds:  (B, T, embed)
            actions: (B, T, A) - actions[t] led to embeds[t]
            init_state: dict with h, z of shape (B, ...)
        Returns:
            dict of stacked tensors with shape (B, T, ...)
        """
        B, T, _ = embeds.shape
        state = init_state
        hs, zs, prior_stats, post_stats = [], [], [], []
        for t in range(T):
            state = self.obs_step(state, actions[:, t], embeds[:, t])
            hs.append(state["h"])
            zs.append(state["z"])
            prior_stats.append(state["prior_stats"])
            post_stats.append(state["post_stats"])
        return {
            "h": torch.stack(hs, dim=1),
            "z": torch.stack(zs, dim=1),
            "prior_stats": torch.stack(prior_stats, dim=1),
            "post_stats": torch.stack(post_stats, dim=1),
        }

    def imagine(self, init_state, actor, horizon):
        """Roll out actor in latent space for `horizon` steps."""
        state = {"h": init_state["h"], "z": init_state["z"]}
        feats, actions, log_probs, entropies = [], [], [], []
        for _ in range(horizon):
            feat = torch.cat([state["h"], state["z"]], dim=-1)
            dist = actor(feat)
            action = dist.rsample()
            log_prob = dist.log_prob(action)
            entropy = -log_prob  # 1-sample entropy estimate
            state = self.img_step(state, action)
            feats.append(torch.cat([state["h"], state["z"]], dim=-1))
            actions.append(action)
            log_probs.append(log_prob)
            entropies.append(entropy)
        return {
            "feat": torch.stack(feats, dim=0),       # (H, N, F)
            "action": torch.stack(actions, dim=0),
            "log_prob": torch.stack(log_probs, dim=0),
            "entropy": torch.stack(entropies, dim=0),
        }

    def kl_loss(self, post_stats, prior_stats, free_bits=1.0):
        post = self._stats_to_dist(post_stats)
        prior = self._stats_to_dist(prior_stats.detach())
        kl_dyn = torch.distributions.kl.kl_divergence(post, prior)
        post_d = self._stats_to_dist(post_stats.detach())
        prior_t = self._stats_to_dist(prior_stats)
        kl_rep = torch.distributions.kl.kl_divergence(post_d, prior_t)
        # Free-bits applied separately to each term, as in DreamerV3.
        kl = 0.5 * kl_dyn.clamp(min=free_bits) + 0.5 * kl_rep.clamp(min=free_bits)
        return kl.mean()


## Reward / continue heads, Actor, Critic

All MLPs use LayerNorm + SiLU. The actor outputs a `TanhNormal` so the 3-d
continuous action is bounded in `[-1, 1]`. The critic predicts `symlog`'d
returns and has an EMA target copy used to bootstrap lambda-returns inside
imagination.


In [ ]:
def mlp(in_size, out_size, hidden, layers=2):
    mods = []
    last = in_size
    for _ in range(layers):
        mods += [nn.Linear(last, hidden), nn.LayerNorm(hidden), nn.SiLU()]
        last = hidden
    mods.append(nn.Linear(last, out_size))
    return nn.Sequential(*mods)


class RewardHead(nn.Module):
    def __init__(self, feat_size, hidden=400):
        super().__init__()
        self.net = mlp(feat_size, 1, hidden)

    def forward(self, feat):
        return self.net(feat).squeeze(-1)


class ContinueHead(nn.Module):
    def __init__(self, feat_size, hidden=400):
        super().__init__()
        self.net = mlp(feat_size, 1, hidden)

    def forward(self, feat):
        return self.net(feat).squeeze(-1)  # logits


class Actor(nn.Module):
    """TanhNormal policy over a 3-d continuous action."""

    def __init__(self, feat_size, action_dim=3, hidden=400, min_std=0.1):
        super().__init__()
        self.net = mlp(feat_size, 2 * action_dim, hidden)
        self.action_dim = action_dim
        self.min_std = min_std

    def forward(self, feat):
        out = self.net(feat)
        mean, std = out.chunk(2, dim=-1)
        mean = 5.0 * torch.tanh(mean / 5.0)
        std = F.softplus(std) + self.min_std
        base = Independent(Normal(mean, std), 1)
        return TransformedDistribution(base, [TanhTransform(cache_size=1)])


class Critic(nn.Module):
    """Value head that predicts symlog'd return."""

    def __init__(self, feat_size, hidden=400):
        super().__init__()
        self.net = mlp(feat_size, 1, hidden)

    def forward(self, feat):
        return self.net(feat).squeeze(-1)


def soft_update(target, source, tau):
    with torch.no_grad():
        for tp, sp in zip(target.parameters(), source.parameters()):
            tp.data.mul_(1.0 - tau).add_(sp.data, alpha=tau)


## Build the modules + world-model loss

`world_model_loss` runs the encoder, the RSSM posterior, and the three heads
(decoder, reward, continue) over a sampled sequence. The loss is

```
L_wm = recon_MSE + reward_NLL(symlog) + continue_BCE + kl_scale * KL(post || prior)
```

with KL free-bits at `1.0 nat` and gradient clipping at `1000`.


In [ ]:
action_dim = 3  # CarRacing-v3: steer, gas, brake

encoder = ConvEncoder(config.embed_size).to(device)
rssm = RSSM(action_dim, config.embed_size, config.hidden_size,
            config.stoch_size, config.mlp_hidden, config.min_std).to(device)
feat_size = config.hidden_size + config.stoch_size
decoder = ConvDecoder(feat_size).to(device)
reward_head = RewardHead(feat_size, config.mlp_hidden).to(device)
continue_head = ContinueHead(feat_size, config.mlp_hidden).to(device)

actor = Actor(feat_size, action_dim, config.mlp_hidden).to(device)
critic = Critic(feat_size, config.mlp_hidden).to(device)
target_critic = Critic(feat_size, config.mlp_hidden).to(device)
target_critic.load_state_dict(critic.state_dict())
for p in target_critic.parameters():
    p.requires_grad_(False)

world_params = (list(encoder.parameters()) + list(rssm.parameters())
                + list(decoder.parameters()) + list(reward_head.parameters())
                + list(continue_head.parameters()))
world_opt = torch.optim.Adam(world_params, lr=config.lr_world, eps=config.eps)
actor_opt = torch.optim.Adam(actor.parameters(), lr=config.lr_actor, eps=config.eps)
critic_opt = torch.optim.Adam(critic.parameters(), lr=config.lr_critic, eps=config.eps)

amp_dtype = torch.bfloat16 if config.use_bf16 and device.type == "cuda" else torch.float32


def world_model_loss(batch):
    obs = batch["obs"]                  # (B, T+1, 3, H, W)
    actions = batch["action"]           # (B, T, A)
    rewards = batch["reward"]           # (B, T)
    continues = batch["continue"]       # (B, T)
    B, Tp1 = obs.shape[:2]
    T = Tp1 - 1

    embeds = encoder(obs)               # (B, T+1, embed)

    init_state = rssm.initial_state(B, device)
    # Use embeds[:, 1:] as posterior evidence for steps 1..T (after action[t]).
    rollout = rssm.observe(embeds[:, 1:], actions, init_state)

    feat = torch.cat([rollout["h"], rollout["z"]], dim=-1)   # (B, T, F)

    recon = decoder(feat)                                    # (B, T, 3, H, W)
    target_imgs = obs[:, 1:]
    recon_loss = F.mse_loss(recon, target_imgs, reduction="none")
    recon_loss = recon_loss.sum(dim=(2, 3, 4)).mean()

    reward_pred = reward_head(feat)
    reward_loss = F.mse_loss(reward_pred, symlog(rewards))

    continue_logits = continue_head(feat)
    continue_loss = F.binary_cross_entropy_with_logits(continue_logits, continues)

    kl = rssm.kl_loss(rollout["post_stats"], rollout["prior_stats"],
                      free_bits=config.kl_free_bits)

    loss = (recon_loss
            + config.reward_scale * reward_loss
            + config.continue_scale * continue_loss
            + config.kl_scale * kl)

    metrics = {
        "wm/loss": loss.detach(),
        "wm/recon": recon_loss.detach(),
        "wm/reward": reward_loss.detach(),
        "wm/continue": continue_loss.detach(),
        "wm/kl": kl.detach(),
    }
    return loss, rollout, metrics


## Imagination + lambda-returns

Starting from every posterior state in the batch, we roll the actor in
latent space for `H` steps. The critic is trained to match TD-lambda
returns computed with the EMA target critic; the actor maximizes those
returns through the reparameterized rollout, plus a small entropy bonus.


In [ ]:
def lambda_return(rewards, values, continues, bootstrap, gamma, lam):
    """Compute TD-lambda returns.

    Args:
        rewards:    (H, N)
        values:     (H, N)   - V(s_t) from target critic at imagined steps
        continues:  (H, N)   - predicted continue prob
        bootstrap:  (N,)     - V(s_{H})
        gamma, lam: scalars
    Returns:
        returns: (H, N)
    """
    H = rewards.shape[0]
    returns = [None] * H
    next_v = bootstrap
    for t in reversed(range(H)):
        delta = rewards[t] + gamma * continues[t] * (1 - lam) * values[t]
        returns[t] = delta + gamma * continues[t] * lam * next_v
        next_v = returns[t]
    return torch.stack(returns, dim=0)


def actor_critic_loss(rollout):
    """Train actor + critic in imagination starting from posterior states."""
    # Flatten (B, T) starts into one batch dim N.
    h = rollout["h"].detach().reshape(-1, config.hidden_size)
    z = rollout["z"].detach().reshape(-1, config.stoch_size)
    init = {"h": h, "z": z}

    imag = rssm.imagine(init, actor, config.horizon)
    feats = imag["feat"]                   # (H, N, F)
    entropies = imag["entropy"]            # (H, N)

    # Reward & continue WITH gradient through `feats` (so the actor learns).
    imag_rewards = symexp(reward_head(feats))
    imag_continues = torch.sigmoid(continue_head(feats))

    # Target values are bootstrapped from the EMA critic; no gradient.
    with torch.no_grad():
        target_values = symexp(target_critic(feats))           # (H, N)

    bootstrap = target_values[-1]
    returns = lambda_return(
        imag_rewards[:-1], target_values[:-1], imag_continues[:-1],
        bootstrap, config.gamma, config.lambda_,
    )                                                          # (H-1, N)

    # Discount mask: cumulative product of imag_continues, shifted by 1.
    with torch.no_grad():
        ones = torch.ones_like(imag_continues[:1])
        cont = torch.cat([ones, imag_continues[:-1]], dim=0)
        discount = torch.cumprod(cont * config.gamma, dim=0) / config.gamma
        weights = discount[:-1]                                # (H-1, N)

    # Critic regression on returns; targets detached so only the critic learns.
    critic_pred = critic(feats[:-1].detach())                  # (H-1, N)
    critic_target = symlog(returns).detach()
    critic_loss = (F.mse_loss(critic_pred, critic_target, reduction="none")
                   * weights).mean()

    # Actor maximizes returns via the reparameterized rollout. Returns have
    # gradient through feats -> actions -> actor params; weights are detached.
    actor_obj = (returns * weights).mean()
    entropy_term = (entropies[:-1] * weights).mean()
    actor_loss = -(actor_obj + config.entropy_scale * entropy_term)

    metrics = {
        "ac/actor_loss": actor_loss.detach(),
        "ac/critic_loss": critic_loss.detach(),
        "ac/return_mean": returns.mean().detach(),
        "ac/entropy": entropies.mean().detach(),
        "ac/imag_reward": imag_rewards.mean().detach(),
    }
    return actor_loss, critic_loss, metrics


## Training loop

Five random warmup episodes seed the buffer, then for every policy decision
we (a) act in the env, (b) buffer the transition, (c) take one gradient step
on the world model and one on the actor/critic. The bf16 autocast wraps both
forward passes; gradient unscaling isn't needed because bf16 has the dynamic
range of fp32. Gradient clipping is applied separately at the world model
(`1000`) and actor/critic (`100`) levels, and the EMA target critic is
updated with a polyak factor of `0.02`.

Set `config.total_env_steps` to a small number first to confirm everything
runs before launching the full training job.


In [ ]:
def collect_episode(env, policy_fn, max_steps=None):
    """Run one episode, returning stacked tensors for the buffer.

    `policy_fn(obs_tensor) -> action_np` of shape (action_dim,).
    """
    obs, _ = env.reset()
    obs_buf = [obs]
    act_buf, rew_buf, cont_buf = [], [], []
    done = False
    steps = 0
    total_reward = 0.0
    while not done:
        action_np = policy_fn(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action_np)
        done = terminated or truncated
        obs_buf.append(next_obs)
        act_buf.append(torch.from_numpy(action_np.astype("float32")))
        rew_buf.append(torch.tensor(float(reward)))
        cont_buf.append(torch.tensor(0.0 if terminated else 1.0))
        total_reward += float(reward)
        obs = next_obs
        steps += 1
        if max_steps is not None and steps >= max_steps:
            break
    return {
        "obs": torch.stack(obs_buf),
        "action": torch.stack(act_buf),
        "reward": torch.stack(rew_buf),
        "continue": torch.stack(cont_buf),
        "total_reward": total_reward,
        "length": steps,
    }


def random_policy(env):
    low = torch.tensor(env.action_space.low, dtype=torch.float32)
    high = torch.tensor(env.action_space.high, dtype=torch.float32)

    def policy(_obs):
        return numpy.random.uniform(low.numpy(), high.numpy()).astype("float32")
    return policy


@torch.no_grad()
def make_dreamer_policy(stochastic=True):
    """Returns a closure with mutable `state` for online interaction."""
    state = {"rssm": rssm.initial_state(1, device),
             "prev_action": torch.zeros(1, action_dim, device=device)}

    def reset():
        state["rssm"] = rssm.initial_state(1, device)
        state["prev_action"] = torch.zeros(1, action_dim, device=device)

    def policy(obs_tensor):
        obs_b = obs_tensor.unsqueeze(0).to(device)
        embed = encoder(obs_b)
        new_state = rssm.obs_step(state["rssm"], state["prev_action"], embed)
        feat = torch.cat([new_state["h"], new_state["z"]], dim=-1)
        dist = actor(feat)
        if stochastic:
            action = dist.sample()
        else:
            # Mode of TanhNormal ~= tanh(mean of base).
            base_mean = dist.base_dist.base_dist.mean
            action = torch.tanh(base_mean)
        state["rssm"] = {"h": new_state["h"], "z": new_state["z"]}
        state["prev_action"] = action
        return action.squeeze(0).cpu().numpy().astype("float32")

    policy.reset = reset
    return policy


def train_step(buffer):
    batch = buffer.sample(config.batch_size, config.seq_len, device)
    if batch is None:
        return None

    with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=amp_dtype, enabled=(device.type == "cuda")):
        wm_loss, rollout, wm_metrics = world_model_loss(batch)
    world_opt.zero_grad(set_to_none=True)
    wm_loss.backward()
    nn.utils.clip_grad_norm_(world_params, config.grad_clip_world)
    world_opt.step()

    with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=amp_dtype, enabled=(device.type == "cuda")):
        actor_loss, critic_loss, ac_metrics = actor_critic_loss(rollout)
    actor_opt.zero_grad(set_to_none=True)
    actor_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), config.grad_clip_ac)
    actor_opt.step()

    critic_opt.zero_grad(set_to_none=True)
    critic_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), config.grad_clip_ac)
    critic_opt.step()

    soft_update(target_critic, critic, config.target_tau)

    metrics = {**wm_metrics, **ac_metrics}
    return {k: v.item() for k, v in metrics.items()}


def evaluate(env, n_episodes=5):
    policy = make_dreamer_policy(stochastic=False)
    rewards, lengths = [], []
    for _ in range(n_episodes):
        policy.reset()
        ep = collect_episode(env, policy)
        rewards.append(ep["total_reward"])
        lengths.append(ep["length"])
    return float(numpy.mean(rewards)), float(numpy.std(rewards)), float(numpy.mean(lengths))


def train(total_env_steps=None):
    env = CarRacingWrapper(config.env_id, config.image_size,
                           config.action_repeat, seed=config.seed)
    eval_env = CarRacingWrapper(config.env_id, config.image_size,
                                config.action_repeat)
    buffer = SequenceReplayBuffer(config.buffer_capacity_episodes,
                                  config.image_size, action_dim)

    if total_env_steps is None:
        total_env_steps = config.total_env_steps

    print(f"Warming up with {config.warmup_episodes} random episodes...")
    rand = random_policy(env)
    for _ in range(config.warmup_episodes):
        ep = collect_episode(env, rand)
        buffer.add_episode(ep["obs"], ep["action"], ep["reward"], ep["continue"])
    env_steps = buffer.total_steps() * config.action_repeat
    print(f"Warmup done. Env steps in buffer: {env_steps}")

    history = {"env_steps": [], "train_reward": [], "eval_reward": [],
               "metrics": []}
    best_eval = -float("inf")
    start_time = time.time()
    pbar = tqdm(total=total_env_steps, initial=env_steps, desc="env steps")

    policy = make_dreamer_policy(stochastic=True)
    while env_steps < total_env_steps:
        policy.reset()
        ep = collect_episode(env, policy)
        ep_steps = ep["length"] * config.action_repeat
        env_steps += ep_steps
        pbar.update(ep_steps)
        buffer.add_episode(ep["obs"], ep["action"], ep["reward"], ep["continue"])
        history["env_steps"].append(env_steps)
        history["train_reward"].append(ep["total_reward"])

        # 1:1 train ratio per *policy* step.
        for _ in range(ep["length"] * config.train_every_steps):
            metrics = train_step(buffer)
            if metrics is not None:
                history["metrics"].append(metrics)

        if (env_steps // config.eval_every_steps
                != (env_steps - ep_steps) // config.eval_every_steps):
            mean_r, std_r, mean_len = evaluate(eval_env, config.eval_episodes)
            history["eval_reward"].append((env_steps, mean_r, std_r))
            elapsed = time.time() - start_time
            tqdm.write(f"[{env_steps:>7}] eval={mean_r:.2f}+/-{std_r:.2f} "
                       f"len={mean_len:.0f} elapsed={elapsed/60:.1f}min")
            if mean_r > best_eval:
                best_eval = mean_r
                torch.save({
                    "encoder": encoder.state_dict(),
                    "rssm": rssm.state_dict(),
                    "decoder": decoder.state_dict(),
                    "reward_head": reward_head.state_dict(),
                    "continue_head": continue_head.state_dict(),
                    "actor": actor.state_dict(),
                    "critic": critic.state_dict(),
                    "target_critic": target_critic.state_dict(),
                    "config": asdict(config),
                    "env_steps": env_steps,
                    "eval_reward": mean_r,
                }, os.path.join(config.log_dir, "best_model.pt"))

    pbar.close()
    env.close()
    eval_env.close()
    return history, buffer


history, buffer = train()


## Training reward curve

Plot the per-episode reward and the periodic deterministic-eval reward.


In [ ]:
matplotlib.pyplot.figure(figsize=(8, 4))
matplotlib.pyplot.plot(history["env_steps"], history["train_reward"],
                       alpha=0.4, label="train (stochastic)")
if history["eval_reward"]:
    es = [e[0] for e in history["eval_reward"]]
    er = [e[1] for e in history["eval_reward"]]
    matplotlib.pyplot.plot(es, er, marker="o", label="eval (deterministic)")
matplotlib.pyplot.xlabel("Environment steps")
matplotlib.pyplot.ylabel("Episode reward")
matplotlib.pyplot.title(f"Dreamer Performance on {config.env_id}")
matplotlib.pyplot.legend()
matplotlib.pyplot.grid(alpha=0.3)
matplotlib.pyplot.show()


## Reconstructions: real frames vs world model output

Sample a sequence from the buffer, run the encoder + RSSM posterior, and
decode each `(h, z)` back into a frame. If training converged the bottom
row should look like a slightly blurred version of the top row.


In [ ]:
@torch.no_grad()
def show_reconstructions(buffer, n_frames=8):
    batch = buffer.sample(1, max(n_frames, 16), device)
    if batch is None:
        print("Buffer too small to sample.")
        return
    obs = batch["obs"]              # (1, T+1, 3, H, W)
    actions = batch["action"]
    embeds = encoder(obs)
    init = rssm.initial_state(1, device)
    rollout = rssm.observe(embeds[:, 1:], actions, init)
    feat = torch.cat([rollout["h"], rollout["z"]], dim=-1)
    recon = decoder(feat).clamp(0.0, 1.0)
    real = obs[0, 1:1 + n_frames].cpu().numpy()
    fake = recon[0, :n_frames].cpu().numpy()
    fig, axes = matplotlib.pyplot.subplots(2, n_frames, figsize=(n_frames * 1.5, 3))
    for i in range(n_frames):
        axes[0, i].imshow(real[i].transpose(1, 2, 0))
        axes[0, i].axis("off")
        axes[1, i].imshow(fake[i].transpose(1, 2, 0))
        axes[1, i].axis("off")
    axes[0, 0].set_title("real", loc="left")
    axes[1, 0].set_title("recon", loc="left")
    matplotlib.pyplot.tight_layout()
    matplotlib.pyplot.show()


show_reconstructions(buffer, n_frames=8)


## Open-loop dream from a real start state

Encode a short context, then let the world model imagine forward without
seeing any further real frames. Actions come from the trained actor. This
is the "dream" — what the agent's internal model thinks the world looks
like when it controls the car.


In [ ]:
@torch.no_grad()
def dream_filmstrip(buffer, context_len=5, dream_len=20):
    batch = buffer.sample(1, context_len + 1, device)
    if batch is None:
        print("Buffer too small to sample.")
        return
    obs = batch["obs"]
    actions = batch["action"]
    embeds = encoder(obs)
    init = rssm.initial_state(1, device)
    rollout = rssm.observe(embeds[:, 1:], actions, init)
    state = {"h": rollout["h"][:, -1], "z": rollout["z"][:, -1]}

    dream_frames = []
    for _ in range(dream_len):
        feat = torch.cat([state["h"], state["z"]], dim=-1)
        dist = actor(feat)
        action = dist.sample()
        state = rssm.img_step(state, action)
        feat = torch.cat([state["h"], state["z"]], dim=-1)
        frame = decoder(feat).clamp(0.0, 1.0)
        dream_frames.append(frame.squeeze(0).cpu().numpy())

    fig, axes = matplotlib.pyplot.subplots(1, dream_len, figsize=(dream_len * 1.2, 1.5))
    for i, f in enumerate(dream_frames):
        axes[i].imshow(f.transpose(1, 2, 0))
        axes[i].axis("off")
    matplotlib.pyplot.suptitle("Dream rollout (decoder output during open-loop imagination)")
    matplotlib.pyplot.tight_layout()
    matplotlib.pyplot.show()


dream_filmstrip(buffer, context_len=5, dream_len=20)


## Side-by-side dream vs real video

Loads the best checkpoint (if one was saved during training), drives the
real env with the deterministic actor, and at every real step also predicts
what the world model thinks the next frame will look like. Saves an MP4.


In [ ]:
def render_dream_vs_real(out_path, max_steps=600):
    ckpt_path = os.path.join(config.log_dir, "best_model.pt")
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        encoder.load_state_dict(ckpt["encoder"])
        rssm.load_state_dict(ckpt["rssm"])
        decoder.load_state_dict(ckpt["decoder"])
        reward_head.load_state_dict(ckpt["reward_head"])
        continue_head.load_state_dict(ckpt["continue_head"])
        actor.load_state_dict(ckpt["actor"])
        critic.load_state_dict(ckpt["critic"])
        print(f"Loaded best checkpoint at env_steps={ckpt.get('env_steps')} "
              f"eval_reward={ckpt.get('eval_reward'):.2f}")
    else:
        print("No best checkpoint found; using current model weights.")

    env = CarRacingWrapper(config.env_id, config.image_size,
                           config.action_repeat)
    obs, _ = env.reset()
    state = rssm.initial_state(1, device)
    prev_action = torch.zeros(1, action_dim, device=device)

    frames = []
    total_reward = 0.0
    with torch.no_grad():
        for _ in range(max_steps):
            obs_b = obs.unsqueeze(0).to(device)
            embed = encoder(obs_b)
            new_state = rssm.obs_step(state, prev_action, embed)
            feat = torch.cat([new_state["h"], new_state["z"]], dim=-1)
            recon = decoder(feat).clamp(0.0, 1.0).squeeze(0).cpu().numpy()

            dist = actor(feat)
            base_mean = dist.base_dist.base_dist.mean
            action = torch.tanh(base_mean)
            action_np = action.squeeze(0).cpu().numpy().astype("float32")

            real_frame = obs.numpy()
            combined = numpy.concatenate([real_frame, recon], axis=2)  # side-by-side
            combined = (combined.transpose(1, 2, 0) * 255).astype("uint8")
            frames.append(combined)

            obs, reward, terminated, truncated, _ = env.step(action_np)
            total_reward += reward
            state = {"h": new_state["h"], "z": new_state["z"]}
            prev_action = action
            if terminated or truncated:
                break

    env.close()
    imageio.mimsave(out_path, frames, fps=20)
    print(f"Wrote {out_path} (reward={total_reward:.2f}, frames={len(frames)})")


video_path = os.path.join(config.log_dir, "dream_vs_real.mp4")
render_dream_vs_real(video_path, max_steps=600)


## Final evaluation

Run the deterministic actor for `eval_episodes` episodes and report mean and
std of the total reward.


In [ ]:
eval_env = CarRacingWrapper(config.env_id, config.image_size, config.action_repeat)
mean_r, std_r, mean_len = evaluate(eval_env, n_episodes=20)
eval_env.close()
print(f"Final eval over 20 episodes: {mean_r:.2f} +/- {std_r:.2f} "
      f"(avg length {mean_len:.0f})")


## Save final weights

Save the most recent (not best) weights for inspection / later loading.


In [ ]:
final_path = os.path.join(config.log_dir, "final_model.pt")
torch.save({
    "encoder": encoder.state_dict(),
    "rssm": rssm.state_dict(),
    "decoder": decoder.state_dict(),
    "reward_head": reward_head.state_dict(),
    "continue_head": continue_head.state_dict(),
    "actor": actor.state_dict(),
    "critic": critic.state_dict(),
    "target_critic": target_critic.state_dict(),
    "config": asdict(config),
}, final_path)
print(f"Saved final model to {final_path}")
